# Gun 1 - LLM ile Otomatik Etiket/Sinif Cikarimi (DOC-24)

OCR ile cikarilan belge metnini bir LLM'e (Claude) vererek belgeye onceden tanimlanmis siniflardan (fatura/sozlesme/dilekce/talep formu vb.) uygun olan birden fazlasini (multi-label) atiyor ve serbest metinli etiketler cikariyoruz. Ayrica "guven" skoru bir esigin altinda kalirsa belge ayri bir "belirsiz" sinifina atilmak yerine `human_review: true` olarak isaretleniyor. `src/classifier.py` modulunu iki asamada test ediyoruz:

1. **Sahte (mock) istemciyle yerel mantik testi** — API anahtari gerektirmez, JSON ayristirma, multi-label filtreleme, fallback ve human_review esik davranisini dogrular.
2. **Gercek API ile uctan uca test** — `data/processed/ocr_real_outputs.json` icindeki 5 gercek belge uzerinde calisir; sonuclar `data/processed/classification_report.json` dosyasina kaydedilir.

In [1]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classifier import classify_document, classify_chunks, DEFAULT_CATEGORIES, DEFAULT_CONFIDENCE_THRESHOLD

print("classifier modulu yuklendi.")
print(f"Varsayilan siniflar: {DEFAULT_CATEGORIES}")
print(f"Varsayilan guven esigi (human_review): {DEFAULT_CONFIDENCE_THRESHOLD}")

classifier modulu yuklendi.
Varsayilan siniflar: ['fatura', 'sözleşme', 'dilekçe', 'talep formu', 'diğer']
Varsayilan guven esigi (human_review): 0.7


## 1. Sahte istemciyle yerel mantik testi

In [2]:
from llm_factory import LLMClient


class FakeClient(LLMClient):
    """classify_document() artik LLMClient.generate() araciligiyla calisiyor
    (DOC-27, Factory entegrasyonu); bu sahte istemci de ayni arayuze uyarak
    API anahtari gerektirmeden yerel mantigi test eder. generate() (loglama/sure
    olcumu iceren) LLMClient'ta somut oldugu icin burada sadece _generate() implemente
    edilir (DOC-28)."""

    model_name = "fake-model"

    def __init__(self, reply):
        self.reply = reply

    def _generate(self, system_prompt, user_message, max_tokens):
        return self.reply


In [3]:
mock_reply = json.dumps({
    "siniflar": ["talep formu"],
    "guven": 0.92,
    "etiketler": ["donanim", "monitor", "ic talep"],
    "gerekce": "Belge bir calisanin ekipman talebi icin yazdigi ic yazi.",
}, ensure_ascii=False)

result = classify_document("Talep Eden: Dila Alpay...", client=FakeClient(mock_reply))
assert result["siniflar"] == ["talep formu"]
assert result["human_review"] is False
print("OK - normal JSON yaniti dogru ayristirildi (guven esigin ustunde, human_review=False):")
print(json.dumps(result, ensure_ascii=False, indent=2))

OK - normal JSON yaniti dogru ayristirildi (guven esigin ustunde, human_review=False):
{
  "siniflar": [
    "talep formu"
  ],
  "guven": 0.92,
  "etiketler": [
    "donanim",
    "monitor",
    "ic talep"
  ],
  "gerekce": "Belge bir calisanin ekipman talebi icin yazdigi ic yazi.",
  "human_review": false
}


In [4]:
fenced_reply = "```json\n" + mock_reply + "\n```"
result_fenced = classify_document("metin", client=FakeClient(fenced_reply))
assert result_fenced["siniflar"] == ["talep formu"]
print("OK - markdown kod blogu icindeki JSON de dogru ayristirildi.")

OK - markdown kod blogu icindeki JSON de dogru ayristirildi.


In [5]:
bad_reply = json.dumps({"siniflar": ["bilinmeyen_sinif"], "guven": 0.5, "etiketler": [], "gerekce": "x"})
result_bad = classify_document("metin", client=FakeClient(bad_reply))
assert result_bad["siniflar"] == ["diğer"]
print(f"OK - listede olmayan tek sinif otomatik olarak fallback'e dustu: {result_bad['siniflar']}")

OK - listede olmayan tek sinif otomatik olarak fallback'e dustu: ['diğer']


In [6]:
multi_reply = json.dumps({
    "siniflar": ["fatura", "sözleşme", "bilinmeyen_sinif"],
    "guven": 0.85,
    "etiketler": ["ek belge"],
    "gerekce": "Belge hem fatura hem de sozlesme eki niteliginde.",
}, ensure_ascii=False)

result_multi = classify_document("metin", client=FakeClient(multi_reply))
assert result_multi["siniflar"] == ["fatura", "sözleşme"]
print(f"OK - gecerli birden fazla sinif korundu, gecersiz olan filtrelendi: {result_multi['siniflar']}")

OK - gecerli birden fazla sinif korundu, gecersiz olan filtrelendi: ['fatura', 'sözleşme']


In [7]:
low_conf_reply = json.dumps({
    "siniflar": ["dilekçe"],
    "guven": 0.4,
    "etiketler": ["belirsiz icerik"],
    "gerekce": "Belge turu net degil.",
}, ensure_ascii=False)

result_low = classify_document("metin", client=FakeClient(low_conf_reply))
assert result_low["human_review"] is True
print(f"OK - guven ({result_low['guven']}) esigin ({DEFAULT_CONFIDENCE_THRESHOLD}) altinda kaldigi icin human_review=True, ayri bir 'belirsiz' sinifi ACILMADI: siniflar={result_low['siniflar']}")

result_custom_threshold = classify_document("metin", client=FakeClient(low_conf_reply), confidence_threshold=0.3)
assert result_custom_threshold["human_review"] is False
print("OK - confidence_threshold parametresiyle esik ozellestirilebiliyor.")

OK - guven (0.4) esigin (0.7) altinda kaldigi icin human_review=True, ayri bir 'belirsiz' sinifi ACILMADI: siniflar=['dilekçe']
OK - confidence_threshold parametresiyle esik ozellestirilebiliyor.


In [8]:
try:
    classify_document("   ")
    print("FAIL - bos text icin ValueError beklenirdi")
except ValueError:
    print("OK - bos text icin ValueError firlatildi.")

OK - bos text icin ValueError firlatildi.


## 2. Gercek belge verisiyle uctan uca test

In [9]:
OCR_PATH = "../data/processed/ocr_real_outputs.json"
with open(OCR_PATH, encoding="utf-8") as f:
    ocr_outputs = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


print(f"{len(ocr_outputs)} belge yuklendi: {list(ocr_outputs.keys())}")

5 belge yuklendi: ['test_talep_01.png', 'test_talep_02.png', 'test_talep_03.png', 'test_talep_04.png', 'test_talep_05.png']


In [10]:
classification_report = {}

for filename, fields in sorted(ocr_outputs.items()):
    document_text = format_document(fields)
    try:
        result = classify_document(document_text)
        classification_report[filename] = result
        print(f"{filename}: siniflar={result['siniflar']}, guven={result.get('guven')}, human_review={result.get('human_review')}, etiketler={result.get('etiketler')}")
    except Exception as e:
        classification_report[filename] = {"hata": str(e)}
        print(f"{filename}: HATA - {e}")

test_talep_01.png: siniflar=['talep formu'], guven=0.95, human_review=False, etiketler=['ek monitör talebi', 'donanım talebi', 'yazılım geliştirme', 'ekipman tahsisi']


test_talep_02.png: siniflar=['talep formu'], guven=0.9, human_review=False, etiketler=['laptop talebi', 'insan kaynaklari', 'ekipman talebi', 'is verimliligi']


test_talep_03.png: siniflar=['talep formu'], guven=0.85, human_review=False, etiketler=['klavye değişimi', 'arıza bildirimi', 'ekipman talebi', 'it talebi']


test_talep_04.png: siniflar=['talep formu'], guven=0.9, human_review=False, etiketler=['yazıcı talebi', 'muhasebe departmanı', 'arıza bildirimi', 'ekipman talebi']


test_talep_05.png: siniflar=['talep formu'], guven=0.9, human_review=False, etiketler=['ek ekran talebi', 'pazarlama departmani', 'donanim talebi', 'verimlilik']


In [11]:
OUT_PATH = "../data/processed/classification_report.json"

if any("hata" not in v for v in classification_report.values()):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(classification_report, f, ensure_ascii=False, indent=2)
    print(f"Sonuclar '{OUT_PATH}' dosyasina kaydedildi.")
else:
    print(
        "Gercek API cagrisi basarisiz oldu (muhtemelen ANTHROPIC_API_KEY "
        ".env dosyasinda tanimli degil). Sonuc dosyasi guncellenmedi; "
        ".env dosyasina gecerli bir anahtar eklenip bu hucre yeniden "
        "calistirilabilir."
    )

Sonuclar '../data/processed/classification_report.json' dosyasina kaydedildi.


## 3. Eşik (guven) kalibrasyonu: farklı sınıflardan ve kasıtlı belirsiz içerikten örnekler

Bölüm 2'deki 5 test belgesinin tamamı aynı sınıfa (talep formu) düşüyor ve güveni 0.85-0.95 arasında — yani `DEFAULT_CONFIDENCE_THRESHOLD = 0.7` değerini gerçekten sınayacak (eşiğin etrafında güven üreten) bir örnek yoktu. Mentörün notu tam olarak bunu işaret ediyor: *"Eşik değerini test sonuçlarına göre belirlemek daha sağlıklı olacaktır."*

Bu bölümde, `classify_document()`'ı doğrudan serbest metinle (görsel/OCR adımı olmadan) çağırarak dört farklı senaryoyu **gerçek API ile** deniyoruz:

1. Net bir fatura metni → yüksek güven bekleniyor.
2. Net bir sözleşme metni → yüksek güven bekleniyor.
3. Hem fatura hem sözleşme içeriği taşıyan karma bir metin → gerçek multi-label (`siniflar` içinde birden fazla eleman) bekleniyor.
4. Kasıtlı olarak belirsiz/az bilgili bir metin → düşük güven ve `human_review=True` bekleniyor.

Sonuçlar `data/processed/classification_calibration_report.json` dosyasına kaydediliyor.

In [12]:
calibration_samples = {
    "fatura_ornegi": (
        "FATURA\n"
        "Fatura No: FTR-2026-00456\n"
        "Fatura Tarihi: 10.08.2026\n"
        "Satici: Ofis Malzemeleri A.S.\n"
        "Alici: Docurag Yazilim Ltd. Sti.\n\n"
        "Urun/Hizmet: 27 inc Monitor x 1 adet\n"
        "Birim Fiyat: 4.500 TL\n"
        "KDV (%20): 900 TL\n"
        "Genel Toplam: 5.400 TL\n\n"
        "Odeme Sekli: Havale/EFT, 15 gun vade."
    ),
    "sozlesme_ornegi": (
        "HIZMET SOZLESMESI\n\n"
        "Isbu sozlesme, bir tarafta 'Yuklenici' olarak anilan Docurag Yazilim Ltd. Sti. "
        "ile diger tarafta 'Musteri' olarak anilan ABC Lojistik A.S. arasinda 01.08.2026 "
        "tarihinde asagidaki sartlarla akdedilmistir.\n\n"
        "Madde 1 - Konu: Yuklenici, Musteri icin belge yonetim yazilimi gelistirecektir.\n"
        "Madde 2 - Sure: Sozlesme 12 ay sureyle gecerlidir.\n"
        "Madde 3 - Fesih: Taraflardan biri 30 gun onceden yazili bildirimle sozlesmeyi feshedebilir.\n\n"
        "Taraflarin imzasi ile yururluge girer."
    ),
    "karma_fatura_sozlesme": (
        "BAKIM VE DESTEK SOZLESMESI EKI - FATURA\n\n"
        "Isbu belge, taraflar arasinda imzalanan yillik bakim ve destek sozlesmesinin "
        "(Sozlesme No: BDS-2026-12) eki niteliginde duzenlenmis fatura belgesidir.\n\n"
        "Sozlesme Kapsami: Sunucu bakim ve destek hizmetleri, aylik periyotlarla saglanir.\n"
        "Fatura No: FTR-2026-00789\n"
        "Fatura Donemi: Agustos 2026\n"
        "Tutar: 12.000 TL + KDV\n\n"
        "Bu fatura, yukarida belirtilen sozlesme hukumleri geregi duzenlenmistir."
    ),
    "belirsiz_ornek": (
        "Sayin ilgili,\n\n"
        "Ekte belirtilen konuyla ilgili geregini rica ederim.\n\n"
        "Saygilarimla."
    ),
    "talep_formu_ornegi": (
        "DONANIM TALEP FORMU\n"
        "Talep Eden: Ayse Yilmaz\n"
        "Departman: Bilgi Islem\n"
        "Talep Tarihi: 15.08.2026\n\n"
        "Talep Edilen Urun: Dizustu bilgisayar (14 inc, 16GB RAM)\n"
        "Gerekce: Mevcut cihaz arizalandi, is surekliligi icin acil ihtiyac var.\n\n"
        "Onaylayan Yonetici: Mehmet Kaya\n"
        "Onay Durumu: Bekliyor"
    ),
    "dilekce_ornegi": (
        "DILEKCE\n\n"
        "Muhatap: Insan Kaynaklari Mudurlugu\n\n"
        "Tarafimca, 01.09.2026 - 05.09.2026 tarihleri arasinda yillik izin kullanilmak "
        "istenmektedir. Bu tarihler arasinda gorevimi devralacak kisi Zeynep Demir olarak "
        "belirlenmistir. Geregini bilgilerinize saygilarimla arz ederim.\n\n"
        "Ad Soyad: Can Ozturk\n"
        "Tarih: 20.08.2026"
    ),
    "diger_ornegi": (
        "TOPLANTI NOTU\n\n"
        "Konu: Haftalik Ekip Senkronizasyonu\n"
        "Tarih: 18.08.2026\n"
        "Katilimcilar: Proje ekibi (5 kisi)\n\n"
        "Gundem Maddeleri:\n"
        "1. Gecen hafta tamamlanan islerin ozeti\n"
        "2. Bu hafta oncelikli gorevler\n"
        "3. Karsilasilan engeller ve cozum onerileri\n\n"
        "Kararlar: Bir sonraki sprint planlamasi Cuma gunu yapilacak."
    ),
    "eksik_taranmis_ornek": (
        "...vam eden surec... belge no: ... tarih okunam...\n"
        "[tarama hatasi: metnin bir kismi okunamadi]\n"
        "... talep ... miktar: ... adet ...\n"
        "imza bolumu net degil, ust kisim kismen kesilmis."
    ),
    "coklu_dilekce_talep": (
        "IZIN VE GOREVLENDIRME DILEKCESI / TALEP FORMU\n\n"
        "Sayin Yetkili,\n\n"
        "Asagida belirtilen tarihler arasinda yillik izin talebimin onaylanmasini ve bu "
        "sure zarfinda ekip arkadasim Ali Veli'ye vekalet verilmesini rica ederim. Ayrica "
        "izin donusu saha calismasi icin kullanilmak uzere bir dizustu bilgisayar talep "
        "ediyorum.\n\n"
        "Izin Tarihleri: 10.09.2026 - 17.09.2026\n"
        "Talep Edilen Ekipman: Dizustu bilgisayar\n"
        "Ad Soyad: Deniz Aydin\n"
        "Tarih: 25.08.2026\n\n"
        "Saygilarimla."
    ),
    "kisa_belirsiz_ornek_2": (
        "Konu: Ek Bilgi\n"
        "Tarih: 22.08.2026\n\n"
        "Gerekli gorulmesi halinde tarafimla iletisime gecilmesi rica olunur."
    ),
}

calibration_report = {}
for name, text in calibration_samples.items():
    result = classify_document(text)
    calibration_report[name] = result
    print(f"{name}: siniflar={result['siniflar']}, guven={result['guven']}, human_review={result['human_review']}")
    print(f"  gerekce: {result.get('gerekce')}\n")

fatura_ornegi: siniflar=['fatura'], guven=0.97, human_review=False
  gerekce: Belge fatura no, tarih, satici-alici bilgileri, urun, kdv ve toplam tutar iceren tipik bir fatura formatindadir.

sozlesme_ornegi: siniflar=['sözleşme'], guven=0.97, human_review=False
  gerekce: Belge iki taraf arasında akdedilen, konu, süre ve fesih maddeleri içeren bir hizmet sözleşmesidir.

karma_fatura_sozlesme: siniflar=['fatura', 'sözleşme'], guven=0.85, human_review=False
  gerekce: Belge hem bir sözleşmeye atıfta bulunan hem de fatura numarası ve tutar bilgisi içeren bir fatura ekidir.

belirsiz_ornek: siniflar=['dilekçe'], guven=0.4, human_review=True
  gerekce: Metin, resmi bir dille 'gereğinin rica edilmesi' ifadesiyle dilekçe formatına benzemekte ancak içerik detayı belirsizdir.

talep_formu_ornegi: siniflar=['talep formu'], guven=0.95, human_review=False
  gerekce: Belge, çalışan tarafından donanım talebini içeren ve yönetici onayı bekleyen bir talep formu niteliğindedir.

dilekce_ornegi: sinifl

In [13]:
CALIBRATION_PATH = "../data/processed/classification_calibration_report.json"
with open(CALIBRATION_PATH, "w", encoding="utf-8") as f:
    json.dump(calibration_report, f, ensure_ascii=False, indent=2)

guven_values = {name: r["guven"] for name, r in calibration_report.items()}
print(f"Kalibrasyon sonuclari kaydedildi -> {CALIBRATION_PATH}\n")
print("guven dagilimi:", guven_values)

assert len(calibration_report["karma_fatura_sozlesme"]["siniflar"]) >= 2, (
    "Karma fatura/sozlesme metni gercek API ile hala tek sinifa dustu - "
    "multi-label uctan uca gercek veriyle dogrulanamadi."
)
print("\nOK - karma metin gercekten birden fazla sinifa atandi (multi-label canli API ile dogrulandi):",
      calibration_report["karma_fatura_sozlesme"]["siniflar"])

Kalibrasyon sonuclari kaydedildi -> ../data/processed/classification_calibration_report.json

guven dagilimi: {'fatura_ornegi': 0.97, 'sozlesme_ornegi': 0.97, 'karma_fatura_sozlesme': 0.85, 'belirsiz_ornek': 0.4, 'talep_formu_ornegi': 0.95, 'dilekce_ornegi': 0.95, 'diger_ornegi': 0.9, 'eksik_taranmis_ornek': 0.3, 'coklu_dilekce_talep': 0.85, 'kisa_belirsiz_ornek_2': 0.4}

OK - karma metin gercekten birden fazla sinifa atandi (multi-label canli API ile dogrulandi): ['fatura', 'sözleşme']


### Kalibrasyon sonucu: esik 0.7'de birakildi (10 orneklik genisletilmis kume ile dogrulandi)

Gercek API sonuclari, orneklem 4'ten 10'a cikarildiktan sonra da ayni net ayrimi gosterdi:

| Ornek | guven | Beklenen davranis | Sonuc |
|---|---|---|---|
| fatura_ornegi | 0.97 | net sinif | dogrulandi |
| sozlesme_ornegi | 0.97 | net sinif | dogrulandi |
| talep_formu_ornegi | 0.95 | net sinif | dogrulandi |
| dilekce_ornegi | 0.95 | net sinif | dogrulandi |
| diger_ornegi | 0.90 | hicbir sinifa uymayan metin -> fallback "diger" | dogrulandi `["diger"]` |
| karma_fatura_sozlesme | 0.85 | gercek multi-label | dogrulandi `["fatura", "sozlesme"]` |
| coklu_dilekce_talep | 0.85 | gercek multi-label (farkli kombinasyon) | dogrulandi `["dilekce", "talep formu"]` |
| belirsiz_ornek | 0.40 | dusuk guven -> human_review | dogrulandi `human_review=True` |
| kisa_belirsiz_ornek_2 | 0.40 | dusuk guven -> human_review | dogrulandi `human_review=True` |
| eksik_taranmis_ornek | 0.30 | yarim/bozuk metin -> en dusuk guven | dogrulandi `human_review=True` |

Net/tek-kategori, coklu-kategori ve fallback "diger" ornekleri **0.85-0.97** bandinda kumelenirken, gercekten belirsiz/az bilgili/hasarli ornekler **0.30-0.40** bandinda kaliyor. Orneklem 4'ten 10'a cikarilmasina ragmen bu iki kume arasindaki bosluk (0.41-0.84) hala doldurulmadi -- yani bu bir tesaduf degil, modelin guven skorunu gercekten "net/anlasilir" ile "belirsiz/eksik" arasinda keskin bir esik gibi kullandigina isaret ediyor. `DEFAULT_CONFIDENCE_THRESHOLD = 0.7` bu boslugun ortasina dusuyor ve genisletilmis orneklemle de dogrulanmis oldu.